# Alpha Genie - Market Data Exploration

This notebook uses the same environment as the backend.

**Make sure you selected kernel: `Alpha Genie (Backend)`**

In [ ]:
# Add backend to path
import sys
sys.path.insert(0, '../backend')

## 2. Using Backend Services

In [ ]:
# Import backend services
from app.services.market_data import market_data_service

# Since services are async, we need to run them properly
import asyncio

async def get_quote(symbol):
    return await market_data_service.get_quote(symbol)

# Get quote
quote = await get_quote("MSFT")
quote

In [ ]:
# Get multiple quotes in parallel
async def get_quotes(symbols):
    return await market_data_service.get_multiple_quotes(symbols)

quotes = await get_quotes(["AAPL", "GOOGL", "MSFT", "AMZN"])

# Display as DataFrame
pd.DataFrame(quotes)[['symbol', 'name', 'price', 'change_percent', 'market_cap']]

## 3. Using LLM Service

In [ ]:
from app.services.llm import llm_service

# Test chat (requires API key in backend/.env)
async def chat(prompt):
    return await llm_service.chat(prompt)

response = await chat("What is a P/E ratio in simple terms?")
print(response)

## 4. Database Access

In [ ]:
from app.core.database import SessionLocal, init_db
from app.models import Document, Portfolio, Holding

# Initialize DB
init_db()

# Create session
db = SessionLocal()

# Query documents
documents = db.query(Document).all()
print(f"Total documents: {len(documents)}")

# Query portfolios
portfolios = db.query(Portfolio).all()
print(f"Total portfolios: {len(portfolios)}")

db.close()

# 4. Test Earning call API 

In [8]:
ninja = 'jrqLVaMNvaDZQVA9cNI0zkVv2fhFqH8nKZyK4NA4'
alpha_avantage_api_key = 'Z7IMC4E0QT0HVTHW'

In [9]:
import requests

def get_earnings_transcript(symbol, quarter, alpha_avantage_api_key):
    earning_call_url = f'https://www.alphavantage.co/query?function=EARNINGS_CALL_TRANSCRIPT&symbol={symbol}&quarter={quarter}&apikey={alpha_avantage_api_key}'
    earning_call_data = requests.get(earning_call_url)
    return earning_call_data.json()

In [10]:
APPLE_2025Q4 = get_earnings_transcript("AAPL", "2025Q4", alpha_avantage_api_key)
#APPLE_2025Q3 = get_earnings_transcript("AAPL", "2025Q3", alpha_avantage_api_key)

In [ ]:
APPLE_2025Q4_full_transcript = '' 

for idx in range(0,len(APPLE_2025Q4['transcript'])):
    APPLE_2025Q4_full_transcript += APPLE_2025Q4['transcript'][idx]['content'] + ' '

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch available: {torch.cuda.is_available()}")

import numpy as np
print(f"NumPy version: {np.__version__}")

from transformers import pipeline
print("Transformers loaded successfully!")

PyTorch version: 2.2.2
PyTorch available: False
NumPy version: 1.26.4
Transformers loaded successfully!
The weather is nice today: general (0.93)
Revenue grew by 20% YoY: financial (0.98)
I like pizza and YoY revenue is 200M dollars: financial (0.97)


In [36]:
financial_sentence_classifier = pipeline("zero-shot-classification", 
                     model="facebook/bart-large-mnli")

/Users/eunjinoh/Desktop/03_aiml/03_Projects/alpha_genie/backend/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline


[{'label': 'positive', 'score': 0.9998133778572083}, {'label': 'neutral', 'score': 0.9997822642326355}, {'label': 'negative', 'score': 0.9877365231513977}]


In [ ]:
APPLE_2025Q4_full_transcript = APPLE_2025Q4_full_transcript.lower()

In [ ]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/eunjinoh/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [30]:
import pandas as pd
earning_class_df = pd.DataFrame({'earning_sentence':earning_report_sentences})

In [ ]:


def detect_financial_sentences(sentence):
    for keyword in financial_keyword:
        if keyword in sentence:
            label = 1
            score = 1.0
            return label, score
    return 0, 0.0


def detect_financial_sentences_lm(sentence):
    labels = ["financial", "general"]
    result = financial_sentence_classifier(sentence, ["financial", "general"])
    label = 1 if result['labels'][0] == "financial" else 0
    score = np.round(result['scores'][0],2)
    return label, score

In [40]:
earning_class_df[['keyword_search_label','keyword_search_score']] = earning_class_df['earning_sentence'].apply(lambda x: pd.Series(detect_financial_sentences(x)))
earning_class_df[['keyword_classified_label','keyword_classified_score']] = earning_class_df['earning_sentence'].apply(lambda x: pd.Series(detect_financial_sentences_lm(x)))

In [42]:
earning_class_df['earning_sentence'][1]

'my name is suhasini chandramouli, director of investor relations.'

In [43]:
earning_financial_sentence = earning_class_df[(earning_class_df['keyword_classified_label'] == 1)&(earning_class_df['keyword_classified_score']>0.90)]

In [57]:
finbert_sentiment_model = BertForSequenceClassification.from_pretrained(
    "ahmedrachid/FinancialBERT-Sentiment-Analysis",
    num_labels=3
)
finbert_tokenizer = BertTokenizer.from_pretrained("ahmedrachid/FinancialBERT-Sentiment-Analysis")

/Users/eunjinoh/Desktop/03_aiml/03_Projects/alpha_genie/backend/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [62]:
def get_finbert_sentiment_scores(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)[0]
    labels = ['negative', 'neutral', 'positive']
    scores = {
        labels[i]: probabilities[i].item() 
        for i in range(3)
    }
    return np.round(scores['positive'],4), np.round(scores['neutral'],4), np.round(scores['negative'],4)


In [63]:
earning_financial_sentence[['finbert_positive_score','finbert_neutral_score','finbert_negative_score']] = earning_financial_sentence['earning_sentence'].apply(lambda x: pd.Series(get_finbert_sentiment_scores(finbert_sentiment_model, finbert_tokenizer,x)))

In [64]:
earning_financial_sentence.head()

,earning_sentence,keyword_serach_label,keyword_search_label,keyword_search_score,keyword_classified_label,keyword_classified_score,finbert_positive_score,finbert_neutral_score,finbert_negative_score
0,"good afternoon, and welcome to the apple q4 fi...",1,1.0,1.0,1.0,0.98,0.0724,0.9256,0.0021
11,you can find a reconciliation of these measure...,1,1.0,1.0,1.0,0.93,0.0002,0.9991,0.0007
15,"today, apple is proud to report $102.5 billion...",1,1.0,1.0,1.0,0.99,0.9998,0.0000,0.0001
17,eps came in at $1.85 setting a september quart...,1,1.0,1.0,1.0,0.94,0.9998,0.0001,0.0002
19,we also set a september quarter revenue record...,1,1.0,1.0,1.0,0.98,0.9998,0.0001,0.0001


In [68]:
positive_score = earning_financial_sentence[['finbert_positive_score']].mean()
neutral_score = earning_financial_sentence[['finbert_neutral_score']].mean()
negative_score = earning_financial_sentence[['finbert_negative_score']].mean()